#### 📃 **Documentation**

##### 💡 **Basic Info**
- Script:     CrossRegionWorkspaceMigration.ipynb
- Purpose:    Script moves workspaces to a different region, while handling Large Semantic Models
- Author:     Paweł Wrona
- Version:    1.0
- Updated:    2026-02-03

##### 📋 **Requirements**
- User running the script must have active **Tenant Admin** role.
- Physical access to Workspaces is not required, it will be resolved if needed.
- Mandatory configuration:
    - **SCRIPT_MODE**: Specifies how Input Workspaces are provided:
        - **0** -> must provide **SOURCE_CAPACITY** to pull all Workspaces.
        - **1** -> You can provide a list of **SOURCE_WORKSPACES** that will be migrated.
    - **TARGET_CAPACITY**: this where Workspaces will be migrated.
    - **ADMIN_UPN**: User Principal Name of person who runs the script. This will be used to resolve the Workspace access when missing.

##### 🔄️ **Algorithm**
Entire Algorithm is split into 3 stages:
- Main Program
- Process Workspaces
- Process Semantic Models

###### **Main Program**
1. Configuration
2. Function Definition
3. Resolve Input Parameters
4. **[Process Workspaces]**
5. Save logs to csv files (if output path provided in configuration)

###### **Process Workspaces**
1. Resolve Workspace Access for a user
2. Search for Large Semantic Models in a Workspace
    1. If Large Semantic Models are found -> **[Process Semantic Models]**. Subprocess generates list of converted Semantic Models, and potential conversion error.
    2. If Large Semantic Models not found -> go to next step
3. Check for conversion_error from [Process Semantic Models]:
    1. Error found -> Workspace is not migrated to a new Capacity
    2. No errors found -> Workspace is migrated to a new Capacity
4. Retrieve the list of converted semantic models from [Process Semantic Models], and uses it to convert semantic models back Large Storage Format
5. Revoke Workspace Access if it was granted in step 1

###### **Process Semantic Models**
1. Try converting Semantic Models to Small Storage Format:
    1. Conversion successful -> Proceed with next step
    2. Conversion failed -> Log failed semantic model and raise conversion_error to be handled by [Process Workspaces] procedure
2. Fetch another Semantic Model until the list is exhausted or there is a failed conversion
3. Check Semantic Model Conversion Status


#### ⚙️ **Configuration**

##### **Import required packages**

In [ ]:
# Import required components
import json, logging
import pyspark.sql.functions as F
import sempy.fabric as fabric
from sempy.fabric.exceptions import FabricHTTPException
import pandas as pd
import time
from pyspark.sql import Row

In [ ]:
# Set up logging configuration (once, ideally at the top of your script)
logging.basicConfig(level=logging.WARNING)  # Or DEBUG, WARNING, etc.

#Instantiate the client
client = fabric.PowerBIRestClient()

##### **Select a Script Mode**
- **0** - Get All Workspaces from a SOURCE_CAPACITY
- **1** - Provide list of Workspaces Manually

In [ ]:
SCRIPT_MODE = 1

##### **Provide Input Parameters**
- **TARGET_CAPACITY** - Capacity GUID where you want to move your Workspaces.
- **SOURCE_CAPACITY** - Capacity GUID from where you want to move your Wokrspaces (relevant only from SCRIPT_MODE = 0).
- **SOURCE_WORKSPACES** - List of Workspace GUIDs that will be moved to a TARGET_CAPACITY (relevant when SCRIPT MODE = 1).
- **OUTPUT_FILE_LOCATION** - When provided, script will write the logs into a *.csv file.
- **ADMIN_UPN** - Used to assign a user with Workspace Access. Needed to perform operations on Semantic Models.

In [ ]:
TARGET_CAPACITY = ""
ADMIN_UPN = ""

# Uncomment the line below, if selected SCRIPT_MODE is 0
# SOURCE_CAPACITY = ""

# Uncomment the line below if selected SCRIPT_MODE is 1
SOURCE_WORKSPACES = []

# Optional step to save the logs outputs
OUTPUT_FILE_LOCATION = ""

### **📃 Function Definitions**

In [ ]:
# Function gets all Capacities as Admin. Needed for Input Parameters validation (source and target capacities),
def get_capacities():
    api_url = "v1.0/myorg/admin/capacities"
    response = client.get(api_url).json()["value"]
    df_capacities = pd.DataFrame(response)[["id", "displayName"]]
    return df_capacities

# Function gets all Workspaces for given Capacity. It is used when SCRIPT_MODE is set to 0.
def get_capacity_workspaces():
    api_url = f"v1.0/myorg/admin/groups?$filter=capacityId eq '{SOURCE_CAPACITY}'&$top=5000"
    response = client.get(api_url).json()
    workspaces = response.get("value", [])
    return workspaces

# Checks correctness of provided workspaces, when SCRIPT_MODE is set to 1.
def get_workspaces_from_list(p_workspace_ids):
    workspace_filter = " or ".join([f"id eq '{wid}'" for wid in p_workspace_ids])
    api_url = f"v1.0/myorg/admin/groups?$filter={workspace_filter}&$top=5000"
    response = client.get(api_url).json()
    workspaces = response.get("value", [])
    return workspaces

# Provides the list of users with access to specified Workspace. It is used to determine if End User must be granted access to Workspace that is being examined.
def get_workspace_users(p_workspace_id):
    api_url = f"v1.0/myorg/admin/groups/{p_workspace_id}/users"
    response = client.get(api_url).json()
    users = response.get("value", [])
    df_users = pd.DataFrame(users)

    if df_users.empty or "graphId" not in df_users.columns:
        return pd.DataFrame()                                               # Function creates empty dataframe if API doesn't return any results

    df_users_filtered = df_users[df_users["graphId"] == ADMIN_UPN]
    return df_users_filtered
    
# Checks if Semantic Models are converted. This propery is hidden on a Workspace level under "capacityMigrationStatus"
# When Semantic Models are still being converted -> status displayed is "PendingMigration"
# When conversion is complete -> status displayed is "Migrated"
def get_semantic_model_migration_status(p_workspace_id):
    api_url = f"v1.0/myorg/admin/groups/{p_workspace_id}"
    response = client.get(api_url).json()
    return response.get("capacityMigrationStatus")

# Functions assigns Admin access to specified Workspace
def assign_workspace_access(p_workspace_id, p_workspace_name):
    api_url = f"v1.0/myorg/admin/groups/{p_workspace_id}/users"
    payload = {
        "principalType": "User",
        "emailAddress": ADMIN_UPN,
        "groupUserAccessRight": "Admin"
    }
    response = client.post(api_url, json=payload)
    if response.status_code in (200, 201):
        print(f"✅ User '{ADMIN_UPN}' assigned to workspace {p_workspace_name}")
        return 1
    else:
        try:
            error = response.json()
        except:
            error = response.text  # fallback if no JSON returned
        
        print(f"❌ Failed to add user '{ADMIN_UPN}' to workspace {p_workspace_name}")
        print(" Response:", error)
        return 0

# Function revokes access to specified Workspace
def revoke_workspace_access(p_workspace_id, p_workspace_name):
    api_url = f"v1.0/myorg/admin/groups/{p_workspace_id}/users/{ADMIN_UPN}"
    response = client.delete(api_url)
    if response.status_code in (200, 201):
        print(f"\n🗑️ User '{ADMIN_UPN}' removed from workspace {p_workspace_name}")
    else:
        try:
            error = response.json()
        except:
            error = response.text  # fallback if no JSON returned
        
        print(f"\n❌ Failed to remove user '{ADMIN_UPN}' from workspace {p_workspace_name}")
        print(" Response:", error)


# Function returns a dataframe with Large Semantic Models, that are found for specified Workspace
def get_premium_storage_datasets(p_workspace_id):
    api_url = f"v1.0/myorg/admin/groups/{p_workspace_id}/datasets"
    response = client.get(api_url).json()
    datasets = response.get("value", [])
    flat_data = [
        {
            "id": ds.get("id"),
            "name": ds.get("name"),
            "targetStorageMode": ds.get("targetStorageMode")
        }
        for ds in datasets
    ]
    df = pd.DataFrame(flat_data)
    df_datasets = df[df["targetStorageMode"] == "PremiumFiles"]
    return df_datasets

# Function converts Semantic Model to a specified storage formatabs
# "Abf" stands for Small Storage Format
# "PremiumFiles" stands for Large Storage Format
def convert_dataset(p_workspace_id, p_dataset_id, p_storage_mode):
    api_url = f"v1.0/myorg/groups/{p_workspace_id}/datasets/{p_dataset_id}"
    payload = {
        "targetStorageMode": p_storage_mode
    }
    try:
        response = client.patch(api_url, json=payload)
        if response.status_code in (200, 201):
            return 1
        else:
            print(f"❌ Unexpected response: {response.status_code} - {response.text}")
            return 0

    # Function is expected to fail, when Semantic Model is already too big to be converted back to Small Storage Format
    except FabricHTTPException as e:
        error_msg = str(e)

        if "larger than 10GB" in error_msg:
            print("❌ Dataset too large for ABF conversion (>10GB). Skipping.")
        else:
            print(f"❌ Conversion API error: {error_msg}")

        return 0

# Final function that moves a Workspace to a TARGET_CAPACITY provided in Configuration Block
def assign_worskapce_to_capacity(p_workspace_id):
    print("\n\n➡️ Assigning Workspace to a new Capacity..")
    api_url = f"v1.0/myorg/admin/groups/{p_workspace_id}"
    payload = {
        "capacityId": TARGET_CAPACITY
    }
    response = client.patch(api_url, json=payload)
    if response.status_code in (200, 201):
        print("✅ Workspace migrated to a new Capacity.")
        return 1
    else:
        print(f"❌ Failed to migrate workspace: {p_workspace_id}")
        return 0

#### **✅ Resolve Input Parameters**
Check **Target** Capacity and (if provided) **Source** Capacity. 

For a Source Capacity Scenario (SCRIPT_MODE == 0), get all Workspaces from Source Capacity and assign them to **SOURCE_WORKSPACES**.

In [ ]:
# Call API to get list of Capacities for your Organization
df_capacities = get_capacities()

# Resolve the TARGET_CAPACITY. If the Capacity ID is correct, script will get its name, that is used later in the code.
df_target_capacity = df_capacities[df_capacities["id"] == TARGET_CAPACITY]
if not df_target_capacity.empty:
    row_target = df_target_capacity.iloc[0]
    target_capacity_name = row_target["displayName"]
    print(f"✅ Target Capacity: {target_capacity_name} was found.")
else:
    print("❌ Target Capacity not found.")

# Source parameters depend on SCRIPT_MODE selected.
if SCRIPT_MODE == 0:

    df_source_capacity = df_capacities[df_capacities["id"] == SOURCE_CAPACITY]
    if not df_source_capacity.empty:
        row_source = df_source_capacity.iloc[0]
        source_capacity_name = row_source["displayName"]
        print(f"✅ Source Capacity: {source_capacity_name} was found. Fetching the list of Workspace from Source Capacity...")
        
        # Script gets all the Workspaces assigned to SOURCE_CAPACITY
        workspaces = get_capacity_workspaces()

    else:
        print(f"❌ Source Capacity was not found.")
        workspaces = None  # ensures variable exists

else:
    # For SCRIPT_MODE == 1, SOURCE_WORKSPACES are taken directly from the list provided in Configuration Block.
    workspaces = get_workspaces_from_list(SOURCE_WORKSPACES)

# Sanity check if Workspaces List has any workspaces assigned to it
if workspaces:
    print(f"✅ Found {len(workspaces)} source workspaces.")
else:
    print(f"❌ Source Workspaces not found. Double check workspace IDs.")

#### **Main Loop**

In [ ]:
# Created empty arrays to log converted and failed Semantic Models
converted_datasets = []
failed_datasets = []

# The Main Loop below goes through all Workspaces in the list
for ws in workspaces:

    # Define Loop Parameters
    access_granted = 0                              # Flag that marks if End User required to grant him/her a Workspace Access. If is used later to revoke it as well.
    conversion_error = 0                            # Errro flag raised when one of the Semantic Models in a Workspace fails to convert to Small Storage Format.
    current_ws_datasets = []                        # Current Semantic Model Batch - array that stores converted semantic models. It resets for each on the list.

    # Assign Workspace attributes to variables
    ws_name = ws["name"]
    ws_id = ws["id"]
    
    print("*" * 100)
    print(f"➡️ Processing Workspace: {ws_name}")
    print("*" * 100)

    # Block checks Workspace level access for End User, and grants it if Access is needed.
    df_users = get_workspace_users(ws_id)
    if df_users.empty:
        print("\n❌ Workspace access missing, granting access... ")
        access_granted = assign_workspace_access(ws_id, ws_name)


    # Get workspace Semantic Models
    print("\n➡️ Processing Large Semantic Models")
    df_datasets = get_premium_storage_datasets(ws_id)
    if not df_datasets.empty:
        
        ds_count = 1                                                                    # Semantic Models counter
        all_ds_count = len(df_datasets)                                                 # Number of all Large Semantic Models in a Workspace
        print(f"⚠️ Found {all_ds_count} Large Semantic Models.")

        for _, ds in df_datasets.iterrows():
            ds_id, ds_name = ds["id"], ds["name"]                                       # Semantic Model ID and Name
            
            print(f"Processing semantic model: {ds_count}/{all_ds_count}...", end="\r")
            
            converted_ds = convert_dataset(ws_id, ds_id, "Abf")                         # Try converting Semantic Model to Small Storage Format
            
            row = {                                                                     # Build a log dataframe row
                "workspace_id": ws_id,
                "workspace_name": ws_name,
                "dataset_id": ds_id,
                "dataset_name": ds_name
            }
        
            if not converted_ds:                                                        # Assign dataframe row to failed_datests if Conversion fails
                print(f"❌ Conversion failed for dataset: {ds_name}. All Semantic Models converted so far will be rolled back to PremiumFiles.")
                failed_datasets.append(row)
                conversion_error = 1
                break
        
            current_ws_datasets.append(row)                                             # For successful conversions, assign row to current workspace batch
            ds_count += 1

        
        # Block of code waits for Semantic Model conversion to finish. Loop stops when migration status is "Migrated".
        print("\n\n➡️ Waiting for Semantic Model conversion to complete.")
        while True:
            status = get_semantic_model_migration_status(ws_id)
            message = f"Semantic Models migration status: {status}"
            print(f"{message:<120}", end="\r")

            if status in ("Migrated"):
                break
            time.sleep(30)  # wait before checking again

        # If scripts finds conversion_error, Workspace is not moved to new Workspace
        if conversion_error:
            print("\n\n⚠️ Conversion error detected. Workspace will not be moved to a new Capacity...")
        else:
            # When there is no conversion_error, current workspace batch info is added to converted_datasets collection
            # As a next step, Workspace is moved to TARGET_CAPACITY
            if not current_ws_datasets == []:
                converted_datasets.extend(current_ws_datasets)
            assign_worskapce_to_capacity(ws_id)

        # Rollback procedure
        # All Semantic Models converted to Abf format are now converted back PremiumFiles
        if current_ws_datasets:

            rev_ds_count = 1
            all_rev_ds_count = len(current_ws_datasets)
            for ds in current_ws_datasets:
                print(f"↩️ Reverting {rev_ds_count}/{all_rev_ds_count} Semantic Models to PremiumFiles...", end="\r")
                ds_id = ds["dataset_id"]
                ds_name = ds["dataset_name"]
                reverted = convert_dataset(ws_id, ds_id, "PremiumFiles")
                if not reverted:
                    print(f"\n❌ Failed to revert dataset: {ds_name}")
                rev_ds_count += 1
        else:
            print("ℹ️ No datasets needed rollback.")

    # Workspaces without Large Semantic Models are ready for migration
    else:
        print("✅ Large Semantic Models not found. Workspace ready for migration.")
        assign_worskapce_to_capacity(ws_id)

    # If End User was granted with Workspace Access, it must be now revoked
    if access_granted == 1:
        revoke_workspace_access(ws_id, ws_name)

    print("\n\n")

In [ ]:
# Build dataframes containing information on conversion progress
# df_converted_datasets - contains all semantic models that were converted correctly, and moved to a new workspace
# df_failed_datasets - semantic models that couldn't be converted to Small Storage Format -> they're the reason why given Workspace was not migrated to a new Capacity

results_columns = [
    "workspace_id",
    "workspace_name",
    "dataset_id",
    "dataset_name"
]

df_converted_datasets = pd.DataFrame(converted_datasets, columns=results_columns)
df_failed_datasets = pd.DataFrame(failed_datasets, columns=results_columns)

# Uncomment below lines to display both dataframes
# display(df_converted_datasets)
# display(df_failed_datasets)

In [ ]:
# Check if End User provided OUTPUT_FILE_LOCATION. If it's available, dataframes are saved to csv files.

if not OUTPUT_FILE_LOCATION == "":
    if not df_converted_datasets.empty:
        df_converted_datasets.to_csv(f"{OUTPUT_FILE_LOCATION}/converted_datasets.csv", index=False)
        print("✅ Converted datasets file saved.")
    else:
        print("ℹ️ No converted datasets to save.")

    if not df_failed_datasets.empty:
        df_failed_datasets.to_csv(f"{OUTPUT_FILE_LOCATION}/failed_datasets.csv", index=False)
        print("✅ Failed datasets file saved.")
    else:
        print("ℹ️ No failed datasets to save.")